In [1]:
# -----######-----###### MAIN IMPORTS -----######-----######
import os
import pandas as pd
from datetime import datetime

# ----- helpers: shared -----
def _tqm_print(step, total, label):
    width = 28
    frac = 0 if total == 0 else step/float(total)
    filled = int(width*frac)
    bar = "█"*filled + " "*(width-filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

def _list_pkls_recursive(root_dir):
    pkl_paths = []
    for r, _, files in os.walk(root_dir):
        for f in files:
            # skip hidden/system artifacts
            if f.startswith("._") or f.startswith(".DS"):
                continue
            if f.lower().endswith(".pkl"):
                pkl_paths.append(os.path.join(r, f))
    return sorted(pkl_paths)

# -----######-----###### MAIN FUNCTION -----######-----######
def _aiff_0810_mergeid_GET_newdf_export(master_pkl_path, search_root, output_dir, id_col="ID"):
    """
    Parameters:
      master_pkl_path: path to master DF .pkl (must contain column 'ID')
      search_root    : root folder to recursively find and concat all .pkl files
      output_dir     : folder to write the resulting _df_aiff_rk_{N}_{yy_mm}.pkl
      id_col         : ID column name to compare (default 'ID')

    Returns:
      saved_path, df_new
    """
    stages = [
        "Scan .pkl files",
        "Read & concat",
        "Load master",
        "Compare IDs",
        "Export result"
    ]

    # Stage 1: scan files
    pkl_files = _list_pkls_recursive(search_root)
    _tqm_print(1, len(stages), stages[0])
    if not pkl_files:
        print("No .pkl files found under:", search_root)

    # Stage 2: read & concat with TQM progress
    dfs = []
    total = len(pkl_files)
    for i, p in enumerate(pkl_files, start=1):
        try:
            df_i = pd.read_pickle(p)
            if id_col in df_i.columns:
                dfs.append(df_i)
            else:
                print(f"Skip (no '{id_col}'): {p}")
        except Exception as e:
            print(f"Skip (read error): {p} -> {e}")
        _tqm_print(i, total if total else 1, "Reading PKLs")

    if dfs:
        df_all = pd.concat(dfs, ignore_index=True)
        # de-duplicate on ID if present
        if id_col in df_all.columns:
            df_all = df_all.drop_duplicates(subset=[id_col], keep="first")
    else:
        df_all = pd.DataFrame(columns=[id_col])
    _tqm_print(2, len(stages), stages[1])

    # Stage 3: load master
    try:
        df_master = pd.read_pickle(master_pkl_path)
    except Exception as e:
        raise RuntimeError(f"Failed to load master DF: {master_pkl_path} -> {e}")
    if id_col not in df_master.columns:
        raise ValueError(f"Master DF missing '{id_col}' column.")
    _tqm_print(3, len(stages), stages[2])

    # Stage 4: compare IDs
    master_ids = set(df_master[id_col].dropna().astype(str))
    # normalize comparison to string to avoid dtype traps
    if id_col in df_all.columns:
        df_all[id_col] = df_all[id_col].astype(str)
    else:
        # if df_all has no ID col at all, then everything is "missing"
        df_all[id_col] = []
    mask_new = ~df_all[id_col].isin(master_ids)
    df_new = df_all.loc[mask_new].copy()
    new_count = len(df_new)
    _tqm_print(4, len(stages), stages[3])
    print(f"Found {new_count} new IDs vs master.")

    # Stage 5: export
    yy = datetime.now().strftime("%y")
    mm = datetime.now().strftime("%m")
    base_name = f"_df_aiff_rk_{new_count}_{yy}_{mm}.pkl"
    os.makedirs(output_dir, exist_ok=True)
    saved_path = os.path.join(output_dir, base_name)
    df_new.to_pickle(saved_path)
    _tqm_print(5, len(stages), stages[4])
    print(f"✅ Exported: {saved_path}")

    return saved_path, df_new
